MODUL 1 — TEXT PROCESSING PIPELINE

Dataset: Ulasan Pengguna Aplikasi Gojek

Notebook ini menggabungkan tahapan pengolahan bahasa alami (Natural Language Processing/NLP), mulai dari membaca dataset, eksplorasi korpus, preprocessing teks, analisis linguistik, N-Gram, hingga Language Model.

Dataset yang digunakan berisi ulasan pengguna aplikasi Gojek. Hasil preprocessing dari praktikum sebelumnya akan dimanfaatkan kembali jika sesuai dengan tahapan yang dibutuhkan.

In [6]:
import pandas as pd

df = pd.read_csv(
    "pertemuan-03/dataset_preprocessed_final.csv",
    low_memory=False
)

print(df.shape)


(225002, 8)


In [9]:
import pandas as pd
from pathlib import Path

dataset_path = Path("pertemuan-03/dataset_preprocessed_final.csv")
df = pd.read_csv(dataset_path, low_memory=False)

print("Jumlah data:", len(df))
print("Kolom:", df.columns.tolist())

Jumlah data: 225002
Kolom: ['userName', 'content', 'score', 'at', 'appVersion', 'clean_text', 'processed_tokens', 'processed_text']


2. Corpus Exploration

Corpus exploration adalah tahap untuk mengenali karakteristik dataset sebelum dilakukan pengolahan teks. Pada tahap ini, dilakukan pemeriksaan jumlah data, jumlah kolom, nama kolom, contoh ulasan, serta kondisi awal data seperti missing value dan duplikasi.

In [35]:
# 2. Corpus Exploration

print("Jumlah dokumen:", len(df))
print("Jumlah kolom:", len(df.columns))

print("\nNama kolom:")
print(df.columns.tolist())

print("\nInformasi dataset:")
df.info()

print("\nMissing value setiap kolom:")
print(df.isnull().sum())

print("\nJumlah baris duplikat:")
print(df.duplicated().sum())

print("\nJumlah content duplikat:")
print(df["content"].duplicated().sum())

Jumlah dokumen: 225002
Jumlah kolom: 8

Nama kolom:
['userName', 'content', 'score', 'at', 'appVersion', 'clean_text', 'processed_tokens', 'processed_text']

Informasi dataset:
<class 'pandas.DataFrame'>
RangeIndex: 225002 entries, 0 to 225001
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   userName          225002 non-null  str  
 1   content           224923 non-null  str  
 2   score             225002 non-null  int64
 3   at                225002 non-null  str  
 4   appVersion        225002 non-null  str  
 5   clean_text        224889 non-null  str  
 6   processed_tokens  225002 non-null  str  
 7   processed_text    221099 non-null  str  
dtypes: int64(1), str(7)
memory usage: 13.7 MB

Missing value setiap kolom:
userName               0
content               79
score                  0
at                     0
appVersion             0
clean_text           113
processed_tokens       0
processed_tex

3. Data Cleaning

Data cleaning dilakukan untuk menghapus ulasan yang kosong dan duplikat. Pembersihan dilakukan pada kolom `content` agar data yang digunakan pada tahap selanjutnya berisi teks yang terisi dan tidak berulang. Dataset awal tetap disimpan pada variabel `df`, sedangkan hasil pembersihan disimpan pada `df_clean`.

In [36]:
# 3. Data Cleaning

jumlah_sebelum = len(df)

df_clean = df.dropna(subset=["content"]).copy()
df_clean = df_clean[
    df_clean["content"].astype(str).str.strip() != ""
]
df_clean = df_clean.drop_duplicates(
    subset=["content"]
).reset_index(drop=True)

print("Jumlah data sebelum cleaning:", jumlah_sebelum)
print("Jumlah data setelah cleaning:", len(df_clean))
print("Jumlah data yang dihapus:", jumlah_sebelum - len(df_clean))
print("Missing value:", df_clean["content"].isna().sum())
print("Duplikat:", df_clean["content"].duplicated().sum())

Jumlah data sebelum cleaning: 225002
Jumlah data setelah cleaning: 137935
Jumlah data yang dihapus: 87067
Missing value: 0
Duplikat: 0


4. Sentence Segmentation

Sentence segmentation adalah proses memisahkan teks menjadi kalimat berdasarkan tanda baca, seperti titik (.), tanda tanya (?), dan tanda seru (!). Tahap ini membantu mengetahui jumlah kalimat dalam setiap ulasan sebelum teks diproses lebih lanjut.

In [37]:
# 4. Sentence Segmentation

import re

def sentence_segmentation(text):
    text = str(text).strip()
    if not text:
        return []

    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [sentence for sentence in sentences if sentence.strip()]

df_clean["sentences"] = df_clean["content"].apply(sentence_segmentation)

jumlah_kalimat = df_clean["sentences"].apply(len).sum()

print("Jumlah dokumen:", len(df_clean))
print("Jumlah kalimat:", jumlah_kalimat)

print("\nContoh hasil sentence segmentation:")
for i in range(min(3, len(df_clean))):
    print(f"\nUlasan {i+1}:")
    print("Teks asli:", df_clean["content"].iloc[i])
    print("Hasil segmentasi:", df_clean["sentences"].iloc[i])

Jumlah dokumen: 137935
Jumlah kalimat: 137935

Contoh hasil sentence segmentation:

Ulasan 1:
Teks asli: akun gopay saya di blok
Hasil segmentasi: ['akun gopay saya di blok']

Ulasan 2:
Teks asli: Lambat sekali sekarang ini bosssku apk gojek gk kaya dulu
Hasil segmentasi: ['Lambat sekali sekarang ini bosssku apk gojek gk kaya dulu']

Ulasan 3:
Teks asli: Kenapa sih dari kemarin sy buka aplikasi gojek malah keluar sendiri terus Saya kasih bintang 2 dulu kalo sudah normal sy kasih bintang 7
Hasil segmentasi: ['Kenapa sih dari kemarin sy buka aplikasi gojek malah keluar sendiri terus Saya kasih bintang 2 dulu kalo sudah normal sy kasih bintang 7']


5. Tokenization

Tokenization adalah proses memecah teks menjadi bagian-bagian kecil yang disebut token, seperti kata, angka, dan tanda baca. Pada tahap ini, teks ulasan dipecah menjadi token tanpa menghapus tanda baca atau mengubah huruf besar dan kecil. Pembersihan dan normalisasi teks dilakukan pada tahap berikutnya.

In [38]:
# Tokenization
def tokenize_text(text):
    return re.findall(r"\w+|[^\w\s]", str(text), flags=re.UNICODE)

df_clean["tokens"] = df_clean["content"].apply(tokenize_text)

# Menghitung jumlah seluruh token
jumlah_token = df_clean["tokens"].apply(len).sum()

print("Jumlah dokumen:", len(df_clean))
print("Jumlah seluruh token:", jumlah_token)

print("\nContoh hasil tokenization:")
for i in range(min(3, len(df_clean))):
    print(f"\nUlasan {i+1}:")
    print("Teks asli:", df_clean["content"].iloc[i])
    print("Token:", df_clean["tokens"].iloc[i])

Jumlah dokumen: 137935
Jumlah seluruh token: 1778124

Contoh hasil tokenization:

Ulasan 1:
Teks asli: akun gopay saya di blok
Token: ['akun', 'gopay', 'saya', 'di', 'blok']

Ulasan 2:
Teks asli: Lambat sekali sekarang ini bosssku apk gojek gk kaya dulu
Token: ['Lambat', 'sekali', 'sekarang', 'ini', 'bosssku', 'apk', 'gojek', 'gk', 'kaya', 'dulu']

Ulasan 3:
Teks asli: Kenapa sih dari kemarin sy buka aplikasi gojek malah keluar sendiri terus Saya kasih bintang 2 dulu kalo sudah normal sy kasih bintang 7
Token: ['Kenapa', 'sih', 'dari', 'kemarin', 'sy', 'buka', 'aplikasi', 'gojek', 'malah', 'keluar', 'sendiri', 'terus', 'Saya', 'kasih', 'bintang', '2', 'dulu', 'kalo', 'sudah', 'normal', 'sy', 'kasih', 'bintang', '7']


In [ ]:

import re
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

# Mengambil stopword Bahasa Indonesia
stop_words = set(stopwords.words("indonesian"))

# Tetap mempertahankan kata negasi
kata_negasi = {"tidak", "bukan", "belum", "jangan", "tanpa"}
stop_words = stop_words - kata_negasi

# Tokenization
def tokenize_text(text):
    return re.findall(r"\w+", str(text).lower())

df_clean["tokens"] = df_clean["content"].apply(tokenize_text)

# Menghapus stopword
df_clean["tokens_no_stopwords"] = df_clean["tokens"].apply(
    lambda tokens: [
        token for token in tokens
        if token not in stop_words
    ]
)

print("Tokenization dan stopword removal selesai!")
print("Contoh token:", df_clean["tokens"].head(3).tolist())
print(
    "Contoh tanpa stopword:",
    df_clean["tokens_no_stopwords"].head(3).tolist()
)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Tokenization dan stopword removal selesai!
Contoh token: [['akun', 'gopay', 'saya', 'di', 'blok'], ['lambat', 'sekali', 'sekarang', 'ini', 'bosssku', 'apk', 'gojek', 'gk', 'kaya', 'dulu'], ['kenapa', 'sih', 'dari', 'kemarin', 'sy', 'buka', 'aplikasi', 'gojek', 'malah', 'keluar', 'sendiri', 'terus', 'saya', 'kasih', 'bintang', '2', 'dulu', 'kalo', 'sudah', 'normal', 'sy', 'kasih', 'bintang', '7']]
Contoh tanpa stopword: [['akun', 'gopay', 'blok'], ['lambat', 'bosssku', 'apk', 'gojek', 'gk', 'kaya'], ['sih', 'kemarin', 'sy', 'buka', 'aplikasi', 'gojek', 'kasih', 'bintang', '2', 'kalo', 'normal', 'sy', 'kasih', 'bintang', '7']]


6. Normalization

Normalization adalah proses menyeragamkan penulisan teks agar lebih konsisten. Pada tahap ini, huruf diubah menjadi huruf kecil (*case folding*), URL dan mention dihapus, serta beberapa bentuk kata tidak baku disesuaikan jika terdapat dalam kamus normalisasi. Tujuannya agar kata yang memiliki bentuk penulisan berbeda dapat diproses dengan lebih konsisten.

In [39]:
# Normalization
import re

# Kamus sederhana untuk normalisasi kata tidak baku
normalisasi_kata = {
    "gk": "tidak",
    "gak": "tidak",
    "nggak": "tidak",
    "ga": "tidak",
    "enggak": "tidak",
    "yg": "yang",
    "aja": "saja",
    "udah": "sudah",
    "dgn": "dengan",
    "krn": "karena",
    "klo": "kalau",
    "kalo": "kalau",
    "sy": "saya",
    "apk": "aplikasi",
}

def normalize_tokens(tokens):
    hasil = []

    for token in tokens:
        # Mengubah huruf menjadi huruf kecil
        token = token.lower()

        # Menghapus URL dan mention
        if token.startswith(("http://", "https://", "www.")):
            continue
        if token.startswith("@"):
            continue

        # Menyesuaikan kata tidak baku
        token = normalisasi_kata.get(token, token)

        hasil.append(token)

    return hasil

df_clean["normalized_tokens"] = df_clean["tokens"].apply(normalize_tokens)

print("Contoh hasil normalization:")
for i in range(min(3, len(df_clean))):
    print(f"\nUlasan {i+1}:")
    print("Sebelum:", df_clean["tokens"].iloc[i])
    print("Sesudah:", df_clean["normalized_tokens"].iloc[i])

Contoh hasil normalization:

Ulasan 1:
Sebelum: ['akun', 'gopay', 'saya', 'di', 'blok']
Sesudah: ['akun', 'gopay', 'saya', 'di', 'blok']

Ulasan 2:
Sebelum: ['Lambat', 'sekali', 'sekarang', 'ini', 'bosssku', 'apk', 'gojek', 'gk', 'kaya', 'dulu']
Sesudah: ['lambat', 'sekali', 'sekarang', 'ini', 'bosssku', 'aplikasi', 'gojek', 'tidak', 'kaya', 'dulu']

Ulasan 3:
Sebelum: ['Kenapa', 'sih', 'dari', 'kemarin', 'sy', 'buka', 'aplikasi', 'gojek', 'malah', 'keluar', 'sendiri', 'terus', 'Saya', 'kasih', 'bintang', '2', 'dulu', 'kalo', 'sudah', 'normal', 'sy', 'kasih', 'bintang', '7']
Sesudah: ['kenapa', 'sih', 'dari', 'kemarin', 'saya', 'buka', 'aplikasi', 'gojek', 'malah', 'keluar', 'sendiri', 'terus', 'saya', 'kasih', 'bintang', '2', 'dulu', 'kalau', 'sudah', 'normal', 'saya', 'kasih', 'bintang', '7']


7. Stopword Removal

Stopword removal adalah proses menghapus kata-kata umum yang dianggap kurang membantu dalam analisis teks, seperti kata penghubung atau kata depan. Pada tahap ini, hasil penghapusan stopword dari preprocessing sebelumnya digunakan jika sudah tersedia.

In [40]:
# Stopword Removal
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

# Mengambil stopword bahasa Indonesia
stopwords_indonesia = set(stopwords.words("indonesian"))

# Mempertahankan kata negasi agar makna ulasan tidak berubah
kata_negasi = {"tidak", "bukan", "belum", "jangan", "tanpa"}
stopwords_indonesia = stopwords_indonesia - kata_negasi

def remove_stopwords(tokens):
    return [
        token for token in tokens
        if token not in stopwords_indonesia and token.isalnum()
    ]

df_clean["tokens_no_stopwords"] = df_clean["normalized_tokens"].apply(remove_stopwords)

print("Contoh hasil stopword removal:")
for i in range(min(3, len(df_clean))):
    print(f"\nUlasan {i+1}:")
    print("Sebelum:", df_clean["normalized_tokens"].iloc[i])
    print("Sesudah:", df_clean["tokens_no_stopwords"].iloc[i])

jumlah_token_sebelum = df_clean["normalized_tokens"].apply(len).sum()
jumlah_token_sesudah = df_clean["tokens_no_stopwords"].apply(len).sum()

print("\nJumlah token sebelum stopword removal:", jumlah_token_sebelum)
print("Jumlah token sesudah stopword removal:", jumlah_token_sesudah)
print("Jumlah token yang dihapus:", jumlah_token_sebelum - jumlah_token_sesudah)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Contoh hasil stopword removal:

Ulasan 1:
Sebelum: ['akun', 'gopay', 'saya', 'di', 'blok']
Sesudah: ['akun', 'gopay', 'blok']

Ulasan 2:
Sebelum: ['lambat', 'sekali', 'sekarang', 'ini', 'bosssku', 'aplikasi', 'gojek', 'tidak', 'kaya', 'dulu']
Sesudah: ['lambat', 'bosssku', 'aplikasi', 'gojek', 'tidak', 'kaya']

Ulasan 3:
Sebelum: ['kenapa', 'sih', 'dari', 'kemarin', 'saya', 'buka', 'aplikasi', 'gojek', 'malah', 'keluar', 'sendiri', 'terus', 'saya', 'kasih', 'bintang', '2', 'dulu', 'kalau', 'sudah', 'normal', 'saya', 'kasih', 'bintang', '7']
Sesudah: ['sih', 'kemarin', 'buka', 'aplikasi', 'gojek', 'kasih', 'bintang', '2', 'normal', 'kasih', 'bintang', '7']

Jumlah token sebelum stopword removal: 1778124
Jumlah token sesudah stopword removal: 1164311
Jumlah token yang dihapus: 613813


In [15]:
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

stop_words = set(stopwords.words("indonesian"))

# Mempertahankan kata negasi
kata_negasi = {"tidak", "bukan", "belum", "jangan", "tanpa"}
stop_words = stop_words - kata_negasi

df_clean["tokens_no_stopwords"] = df_clean["tokens"].apply(
    lambda tokens: [
        token for token in tokens
        if token not in stop_words
    ]
)

print("Stopword removal selesai!")
print(df_clean["tokens_no_stopwords"].head(3).tolist())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Stopword removal selesai!
[['akun', 'gopay', 'blok'], ['lambat', 'bosssku', 'apk', 'gojek', 'gk', 'kaya'], ['sih', 'kemarin', 'sy', 'buka', 'aplikasi', 'gojek', 'kasih', 'bintang', '2', 'kalo', 'normal', 'sy', 'kasih', 'bintang', '7']]


8. Stemming

Stemming adalah proses mengubah kata berimbuhan menjadi kata dasar. Contohnya, kata "berjalan" menjadi "jalan" dan "aplikasinya" menjadi "aplikasi". Pada tahap ini, stemming dilakukan menggunakan library Sastrawi yang mendukung bahasa Indonesia. Tujuannya untuk menyamakan bentuk kata agar lebih mudah dianalisis.

In [ ]:
# Stemming versi lebih efisien
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Mengumpulkan kata unik dari seluruh dokumen
kata_unik = set(
    token
    for tokens in df_clean["tokens_no_stopwords"]
    for token in tokens
)

print("Jumlah kata unik:", len(kata_unik))

# Melakukan stemming satu kali untuk setiap kata unik
kamus_stemming = {
    kata: stemmer.stem(kata)
    for kata in kata_unik
}

# Menggunakan hasil stemming yang sudah disimpan
df_clean["stemmed_tokens"] = df_clean["tokens_no_stopwords"].apply(
    lambda tokens: [kamus_stemming[kata] for kata in tokens]
)

print("Stemming selesai!")
print("Contoh hasil:", df_clean["stemmed_tokens"].head(3).tolist())

Jumlah kata unik: 75685


In [ ]:

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

total_data = len(df_clean)
hasil_stemming = []

print(f"Mulai stemming {total_data:,} dokumen...")

for i, tokens in enumerate(df_clean["tokens_no_stopwords"], start=1):
    hasil_stemming.append([stemmer.stem(token) for token in tokens])

    if i % 500000 == 0 or i == total_data:
        print(f"Sudah diproses: {i:,} / {total_data:,} dokumen")

df_clean["stemmed_tokens"] = hasil_stemming

print("Stemming selesai!")
print("Contoh hasil:", df_clean["stemmed_tokens"].head(3).tolist())


Mulai stemming 137,935 dokumen...


In [13]:

# Membuat data bersih
df_clean = df.dropna(subset=["content"]).copy()

# Menghapus teks yang kosong
df_clean = df_clean[
    df_clean["content"].astype(str).str.strip() != ""
]

# Menghapus ulasan duplikat
df_clean = df_clean.drop_duplicates(
    subset=["content"]
).reset_index(drop=True)

print("Jumlah data sebelum cleaning:", len(df))
print("Jumlah data setelah cleaning:", len(df_clean))
print("Missing value:", df_clean["content"].isna().sum())
print("Duplikat:", df_clean["content"].duplicated().sum())


Jumlah data sebelum cleaning: 225002
Jumlah data setelah cleaning: 137935
Missing value: 0
Duplikat: 0


9. POS Tagging

POS Tagging adalah proses memberikan label jenis kata, seperti kata benda (*noun*), kata kerja (*verb*), dan kata sifat (*adjective*). Proses ini membantu memahami fungsi setiap kata dalam ulasan pengguna Gojek.

In [3]:
# 9. POS Tagging

import spacy_udpipe

spacy_udpipe.download("id")


nlp = spacy_udpipe.load("id")

teks = "aplikasi gojek sangat bagus"
doc = nlp(teks)

print("Hasil POS Tagging:")
for token in doc:
    print(
        token.text,
        "| POS:", token.pos_,
        "| Dependency:", token.dep_,
        "| Head:", token.head.text
    )

URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1006)>

In [4]:
%pip install --upgrade certifi spacy-udpipe

Note: you may need to restart the kernel to use updated packages.


In [26]:
# Memuat model UDPipe bahasa Indonesia
import spacy_udpipe

spacy_udpipe.download("id")
nlp = spacy_udpipe.load("id")

# Contoh POS Tagging
teks = "aplikasi gojek sangat bagus"

doc = nlp(teks)

print("Hasil POS Tagging:")
for token in doc:
    print(token.text, "|", token.pos_, "|", token.dep_, "|", token.head.text)

URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1006)>